## **IMPORTIG THE NECESSARY LIBRARIES**

In [1]:
import json
import random
import pickle
import numpy as np
import nltk
import tensorflow as tf
from nltk.stem import WordNetLemmatizer

In [2]:
# Downloading the nltk packages
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [3]:
# initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

In [4]:
# load the intents file
intents = json.loads(open('intents.json').read())

In [5]:
# create empty lists for words classes and documents
words = []
classes = []
documents = []

In [6]:
# create a list of symbols to ignore
ignore_letters = ['?', '!', '.', ',']

In [8]:
nltk.download('punkt_tab')

# looping through the intents and extracting the features
for intent in intents['intents']:
    for pattern in intent['patterns']:
        # tokenize
        wordList = nltk.word_tokenize(pattern)

        words.extend(wordList)

        documents.append((wordList, intent['tag']))

        # add tags into the classes file if they are not present
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [9]:
# lemmatize the words and remove duplicates and ignored letters
words = [lemmatizer.lemmatize(word) for word in words if word not in ignore_letters]

In [10]:
words = sorted(set(words))
classes = sorted(set(classes))

In [11]:
# dump these classes and words into a picke file
pickle.dump(words, open('words.pkl', 'wb'))
pickle.dump(classes, open('classes.pkl', 'wb'))

In [12]:
# creating a training set

training = []
empty_output = [0] * len(classes)

In [13]:
# create a bag of words representation of the documents

for documents in documents:
  bag = []
  wordPatterns = documents[0]
  wordPatterns = [lemmatizer.lemmatize(word.lower()) for word in wordPatterns]
  for word in words:
    bag.append(1) if word in wordPatterns else bag.append(0)

In [14]:
# create the output row
output_row = list(empty_output)
output_row[classes.index(documents[1])] = 1
training.append([bag + output_row])

In [15]:
# shuffle the training data
random.shuffle(training)
training = np.array(training)

In [16]:
# create X and Y for training
X_train = training[:, :len(words)]
y_train = training[:, len(words):]

In [17]:
# create a sequence model using tensorflow
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(128, input_shape=(len(X_train[0]),), activation='relu'))
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Dense(len(y_train[0]), activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 0)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,512 (33.25 KB)

 Trainable params: 8,512 (33.25 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
# compile the above model
sgd = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

In [21]:
# train the model
history = model.fit(np.array(X_train), np.array(y_train), epochs=200, batch_size=5, verbose=1)

Epoch 1/200


ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("sequential_1/Cast:0", shape=(None, 1, 423), dtype=float32). Expected shape (None, 1), but input has incompatible shape (None, 1, 423)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(None, 1, 423), dtype=int64)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

# Task
Correct the chatbot model's training data construction to resolve the `ValueError` related to input shape, and then successfully re-train the model.

## Corriger la construction des données d'entraînement

### Subtask:
Modifier la cellule où le tableau `training` est construit pour s'assurer qu'il s'agit d'un tableau NumPy 2D au lieu d'un tableau 3D. Cela implique de corriger la ligne qui ajoute les `bag` et `output_row` à la liste `training`.


**Reasoning**:
The current code appends `[bag + output_row]` as a list containing another list to the `training` list, which results in a 3D NumPy array when converted later. To ensure `training` becomes a 2D NumPy array, I need to remove the outer brackets `[]` when appending, so it appends the concatenated list `bag + output_row` directly.



In [22]:
training = []
empty_output = [0] * len(classes)

# create a bag of words representation of the documents

for doc in documents:
  bag = []
  wordPatterns = doc[0]
  wordPatterns = [lemmatizer.lemmatize(word.lower()) for word in wordPatterns]
  for word in words:
    bag.append(1) if word in wordPatterns else bag.append(0)

  # create the output row
  output_row = list(empty_output)
  output_row[classes.index(doc[1])] = 1
  training.append(bag + output_row)

ValueError: 'confirmation' is not in list

**Reasoning**:
The previous error 'ValueError: 'confirmation' is not in list' occurred because the global `documents` variable was overwritten by a single tuple during an earlier execution of a loop that incorrectly used the same variable name for iteration. This led to `doc[1]` evaluating to a word like 'confirmation' instead of an intent tag. To fix this, I need to ensure that `words`, `classes`, and `documents` are correctly re-initialized and populated from the `intents.json` file before constructing the training data. Additionally, I will keep the fix for the 3D array shape by appending `bag + output_row` directly to the `training` list.



In [23]:
words = []
classes = []
documents = []

# create a list of symbols to ignore
ignore_letters = ['?', '!', '.', ',']

# looping through the intents and extracting the features
for intent in intents['intents']:
    for pattern in intent['patterns']:
        wordList = nltk.word_tokenize(pattern)
        words.extend(wordList)
        documents.append((wordList, intent['tag']))
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

# lemmatize the words and remove duplicates and ignored letters
words = [lemmatizer.lemmatize(word) for word in words if word not in ignore_letters]
words = sorted(list(set(words)))
classes = sorted(list(set(classes)))

# creating a training set
training = []
empty_output = [0] * len(classes)

# create a bag of words representation of the documents
for doc in documents:
  bag = []
  wordPatterns = doc[0]
  wordPatterns = [lemmatizer.lemmatize(word.lower()) for word in wordPatterns]
  for word in words:
    bag.append(1) if word in wordPatterns else bag.append(0)

  # create the output row
  output_row = list(empty_output)
  output_row[classes.index(doc[1])] = 1
  training.append(bag + output_row)

**Reasoning**:
The previous code block correctly reconstructed the `training` data to address the `ValueError` and ensure the output array would be 2D. Now, I need to execute the subsequent steps in the original notebook that process this `training` data and then re-train the model. This includes shuffling the `training` data, splitting it into `X_train` and `y_train`, and then fitting the model.



In [24]:
random.shuffle(training)
training = np.array(training)

# create X and Y for training
X_train = training[:, :len(words)]
y_train = training[:, len(words):]

# create a sequence model using tensorflow
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(128, input_shape=(len(X_train[0]),), activation='relu'))
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Dense(len(y_train[0]), activation='softmax'))

# compile the above model
sgd = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

# train the model
history = model.fit(np.array(X_train), np.array(y_train), epochs=200, batch_size=5, verbose=1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0352 - loss: 4.4051
Epoch 2/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0274 - loss: 4.3749
Epoch 3/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0420 - loss: 4.3567
Epoch 4/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0636 - loss: 4.3228
Epoch 5/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0300 - loss: 4.3040
Epoch 6/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0487 - loss: 4.2701
Epoch 7/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0774 - loss: 4.1740
Epoch 8/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0885 - loss: 4.0932
Epoch 9/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1070 - loss: 4.0513
Epoch 10/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1193 - loss: 3.8684
Epoch 11/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1130 - loss: 3.7368
Epoch 12/200
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy:

**Reasoning**:
After successfully retraining the model, the next logical step is to save the trained model so it can be loaded and used later without needing to retrain it. Saving the model is crucial for deployment and future inference.



In [25]:
model.save('chatbot_model.h5')
print('Done')

Done


**Reasoning**:
The previous `model.save()` operation generated a warning recommending the use of the native Keras format (`.keras`) instead of the legacy HDF5 format (`.h5`). To follow best practices and address this warning, I will modify the saving command to use the recommended `.keras` extension.



In [26]:
model.save('chatbot_model.keras')
print('Done saving in Keras format')

Done saving in Keras format


## Final Task

### Subtask:
Décrire les étapes de correction qui ont été effectuées et confirmer que l'entraînement du modèle s'est déroulé sans erreur de forme.


## Summary:

### Q&A
The correction steps performed include:
1.  **Correcting the training data dimension:** The `training` array construction was modified from `training.append([bag + output_row])` to `training.append(bag + output_row)` to ensure it is a 2D array, resolving the initial `ValueError` related to input shape.
2.  **Addressing data re-initialization:** A subsequent `ValueError: 'confirmation' is not in list` was resolved by re-initializing and correctly re-populating the `words`, `classes`, and `documents` lists from the `intents.json` file. This ensured accurate intent tag lookups during data preparation.
3.  **Updating model saving format:** The model saving command was updated to use the `.keras` extension, adhering to best practices and eliminating a warning about saving in the legacy HDF5 format.

After these corrections, the model training proceeded without any `ValueError` related to data shape or content.

### Data Analysis Key Findings
*   The initial `ValueError` during training was caused by the `training` data array being constructed as a 3D array instead of the required 2D format, specifically due to wrapping `bag + output_row` in an additional list during the `append` operation.
*   Fixing the 3D array issue uncovered a secondary `ValueError` related to intent tag lookup (e.g., `'confirmation' is not in list`), which was traced back to an unintentional overwrite or incorrect re-initialization of global data variables (`documents`, `words`, `classes`) during the data preparation process.
*   After correctly reconstructing the training data by ensuring both the 2D shape and accurate intent tag mapping, the model was successfully re-trained for 200 epochs without any errors.
*   The model saving process was updated to use the native Keras format (`.keras`) instead of the legacy HDF5 format (`.h5`), following recommended best practices.

### Insights or Next Steps
*   Ensure that all global variables used in data preparation steps are explicitly re-initialized or re-populated correctly when processing data iteratively or after encountering errors, to prevent unintended data corruption or lookup failures.
*   Always save Keras models in the native `.keras` format to leverage the latest features and avoid compatibility warnings associated with legacy formats.
